In [ ]:
import sys, os
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))  # scrape_code/
sys.path.insert(0, os.path.join(_ROOT, 'scrapers'))
sys.path.insert(0, os.path.join(_ROOT, 'utils'))

In [1]:
import pandas as pd 
import requests 
import numpy as np 
from bs4 import BeautifulSoup
import json, os, time, pdb 
import sys 
import warnings, logging 
import itertools 
from tqdm import tqdm
from argparse import ArgumentParser
from util_funcs import * 
from selenium import webdriver 
from selenium.webdriver.chrome.options import Options 
from selenium.webdriver.common.by import By 
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from scraper import scrape_model

def close_out_driver(wd):
    wd.close()
    wd.quit()


headers = {'User-Agent': 
           'Mozilla/5.0 (X11; Linux x86_64)'+\
            'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

In [6]:
league_dict = {
    'URC': 270557, 
    'Prem': 267979,
    'T14': 270559,
    'RWC': 164205,
    'RChamp': 244293,
    'ChampCup': 271937,
    'ChallCup': 272073,  
    'SR': 242041, 
    'PNCup': 256449,
    'SixNat': 180659
}

league = league_dict['URC']
season = 2022

match_scrape = scrape_model(
    league_set=[league], 
    season_set=[season], 
    update_type='single team',
    date_set=None
    # driver_type='testing'
)

test_data_pull = match_scrape.gather_season_teams(
    league,
    season=season
)

print("team data for season pulled")
team_data_join_back = test_data_pull[
    ['game_id', 'date', 'competition', 'season', 'stadium']
]

team data for season pulled


In [7]:
test_data_pull.game_id.nunique()

183

In [8]:
test_data_pull.head()

,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,"Mon, Dec 26",Munster,Leinster,MUN,LEI,/rugby/match/_/gameId/599459/league/270557,19 - 20,19,20,United Rugby Championship,"Thomond Park, Limerick",599459,270557,2022
1,"Sat, Dec 3",Leinster,Ulster,LEI,ULS,/rugby/match/_/gameId/599450/league/270557,38 - 29,38,29,United Rugby Championship,"RDS Arena, Dublin",599450,270557,2022
2,"Sat, Nov 26",Leinster,Glasgow Warriors,LEI,GLA,/rugby/match/_/gameId/599440/league/270557,40 - 5,40,5,United Rugby Championship,"RDS Arena, Dublin",599440,270557,2022
3,"Fri, Oct 28",Scarlets,Leinster,SCA,LEI,/rugby/match/_/gameId/599428/league/270557,5 - 35,5,35,United Rugby Championship,"Parc y Scarlets, Llanelli",599428,270557,2022
4,"Sat, Oct 22",Leinster,Munster,LEI,MUN,/rugby/match/_/gameId/599425/league/270557,27 - 13,27,13,United Rugby Championship,"Aviva Stadium, Dublin",599425,270557,2022


In [9]:
test_data_pull.to_csv('formed_data/game_schedule_data/URC_schedule_data_{}.csv'.format(season), index=False)

In [4]:
test_data_pull

,date,home_team,away_team,home_team_abbr,away_team_abbr,game_link,score,home_score,away_score,competition,stadium,game_id,league_id,season
0,"Fri, Dec 29",Bath Rugby,Wasps,BAT,WASPS,/rugby/match/_/gameId/291620/league/267979,26 - 31,26,31,Gallagher Prem,"Recreation Ground, Bath",291620,267979,2017
1,"Sat, Dec 23",Wasps,Gloucester Rugby,WASPS,GLO,/rugby/match/_/gameId/291618/league/267979,49 - 24,49,24,Gallagher Prem,"Coventry Building Society Arena, Coventry",291618,267979,2017
2,"Sat, Dec 2",Wasps,Leicester Tigers,WASPS,LEI,/rugby/match/_/gameId/291612/league/267979,32 - 25,32,25,Gallagher Prem,"Coventry Building Society Arena, Coventry",291612,267979,2017
3,"Sun, Nov 26",London Irish,Wasps,IRISH,WASPS,/rugby/match/_/gameId/291606/league/267979,13 - 17,13,17,Gallagher Prem,"Madejski Stadium, Reading",291606,267979,2017
4,"Sat, Nov 18",Wasps,Newcastle Falcons,WASPS,NEW,/rugby/match/_/gameId/291599/league/267979,40 - 10,40,10,Gallagher Prem,"Coventry Building Society Arena, Coventry",291599,267979,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218,"Fri, Sep 15",Sale Sharks,London Irish,SAL,IRISH,/rugby/match/_/gameId/291568/league/267979,36 - 7,36,7,Gallagher Prem,"Salford City Stadium, Salford",291568,267979,2017
224,"Fri, Apr 7",Sale Sharks,Worcester Warriors,SAL,WORCS,/rugby/match/_/gameId/290100/league/267979,36 - 26,36,26,Gallagher Prem,"Salford City Stadium, Salford",290100,267979,2017
231,"Sun, Jan 1",Sale Sharks,Bristol Rugby,SAL,BRI,/rugby/match/_/gameId/290063/league/267979,23 - 24,23,24,Gallagher Prem,"Salford City Stadium, Salford",290063,267979,2017
233,"Fri, Dec 22",Worcester Warriors,London Irish,WORCS,IRISH,/rugby/match/_/gameId/291619/league/267979,23 - 8,23,8,Gallagher Prem,"Sixways, Worcester",291619,267979,2017


In [9]:
test_data_pull['game_id'].iloc[1]

'291618'

In [6]:
test_data_pull['league_id'].iloc[0]

'267979'

In [ ]:
match_scrape.get_match_stats(
    game_id='291620',
    league_id='267979'
)

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_tackles,away_tackle_perc,home_red_cards,home_yellow_cards,home_free_kicks_con,away_red_cards,away_yellow_cards,away_free_kicks_con,home_penalties,away_penalties
0,291620,267979,Bath Rugby,26,Wasps,31,4,5,3,3,...,115/133,86%,0,0,1,0,2,1,4,11


In [10]:
match_scrape.get_match_stats(
    game_id='291618',
    league_id='267979'
)

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_tackles,away_tackle_perc,home_red_cards,home_yellow_cards,home_free_kicks_con,away_red_cards,away_yellow_cards,away_free_kicks_con,home_penalties,away_penalties
0,291618,267979,Wasps,49,Gloucester Rugby,24,6,4,5,2,...,112/134,84%,0,0,0,0,0,2,8,9


In [12]:
# URL declaration, scraping, and initial parsing to get the tables on the page 
url = 'https://www.espn.com/rugby/matchstats/_/gameId/{}/league/{}'.format(
    '291618', '267979'
)
response = requests.get(url, headers=headers).content
soup = BeautifulSoup(response, 'html.parser')
tables = soup.find_all('table')


In [13]:
top_bar = soup.find(class_='competitors')

In [14]:
four_tables = soup.find_all(
    class_='sub-module equal-height countChartList height-reset'
)
check_top_largeLabels = soup.find_all(
    class_="stat-graph compareLineGraph twoTeam largeLabels"
)
stacked_rls = soup.find_all(class_='stacked-rl')

In [15]:
match_event = four_tables[0].find('tbody')

In [5]:
game_dfs = []
for game in tqdm(range(len(test_data_pull.iloc[:10]))):
    time.sleep(2)
    try:
        if type(test_data_pull['game_id'].iloc[game]) == type('tester'):
            game_dfs.append(match_scrape.get_match_stats(
                game_id=test_data_pull['game_id'].iloc[game],
                league_id=test_data_pull['league_id'].iloc[game],
            ))
            # time.sleep(3)
    except Exception as e: 
        print(e)
        # pdb.set_trace()

 10%|█         | 1/10 [00:02<00:22,  2.46s/it]

'NoneType' object has no attribute 'find'


 30%|███       | 3/10 [00:07<00:17,  2.55s/it]

'NoneType' object has no attribute 'find'


 40%|████      | 4/10 [00:09<00:15,  2.52s/it]

...

'NoneType' object has no attribute 'find'


 90%|█████████ | 9/10 [00:22<00:02,  2.61s/it]

'NoneType' object has no attribute 'find'


100%|██████████| 10/10 [00:25<00:00,  2.54s/it]

'NoneType' object has no attribute 'find'


In [7]:
len(game_dfs)

1

In [4]:
all_teams_df = pd.concat(game_dfs, axis=0)
all_teams_df = all_teams_df.merge(team_data_join_back, how='left', on='game_id')
all_teams_df = match_scrape.clean_match_stats(all_teams_df)


In [5]:
all_teams_df.iloc[:5]

,game_id,league_id,home_team,home_team_score,away_team,away_team_score,home_tries,away_tries,home_conversions,away_conversions,...,away_rucks_won_percent,away_mauls_won_num,away_mauls_won_total_num,away_mauls_won_percent,away_scrum_num,away_scrum_total_num,away_scrum_won_percent,away_lineout_num,away_lineout_total_num,away_lineout_won_percent
0,600264,180659,France,35,Scotland,16,4,1,3,1,...,0.96,5,5,1.00,5,7,0.71,15,16,0.93
1,600259,180659,Ireland,27,France,42,3,5,3,4,...,0.93,8,9,0.88,3,3,1.00,12,12,1.00
2,600258,180659,Italy,24,France,73,3,11,3,9,...,0.99,7,8,0.87,5,5,1.00,15,15,1.00
3,600254,180659,England,26,France,25,4,3,3,2,...,0.94,4,4,1.00,5,5,1.00,9,9,1.00
4,600250,180659,France,43,Wales,0,7,0,4,0,...,0.97,2,3,0.66,9,10,0.90,7,8,0.87


In [1]:
player_dfs = []
for game in tqdm(range(len(all_teams_df.iloc[:5]))):
    try:
        player_dfs.append(
            match_scrape.get_player_stats(
                game_id=all_teams_df.iloc[game]['game_id'],
                league_id=all_teams_df.iloc[game]['league_id'])
        )
    except:
        print("Skipping game {}".format(all_teams_df.iloc[game]['game_id']))

NameError: name 'tqdm' is not defined

In [6]:
match_scrape.start_up_driver()

In [ ]:
match_scrape == None

False

In [6]:
all_teams_df.iloc[0]['game_id']

'597390'

In [7]:
match_scrape.get_player_stats(
    game_id=all_teams_df.iloc[0]['game_id'],
    league_id=all_teams_df.iloc[0]['league_id']
    )

[<selenium.webdriver.remote.webelement.WebElement (session="0d347d412f2ce077656e940baf00946a", element="f.E5D17DD505834005EF496BBDFC6DDF12.d.06C69BAA94BA824E74CEB1B6B2D6B90A.e.28")>, <selenium.webdriver.remote.webelement.WebElement (session="0d347d412f2ce077656e940baf00946a", element="f.E5D17DD505834005EF496BBDFC6DDF12.d.06C69BAA94BA824E74CEB1B6B2D6B90A.e.29")>, <selenium.webdriver.remote.webelement.WebElement (session="0d347d412f2ce077656e940baf00946a", element="f.E5D17DD505834005EF496BBDFC6DDF12.d.06C69BAA94BA824E74CEB1B6B2D6B90A.e.30")>, <selenium.webdriver.remote.webelement.WebElement (session="0d347d412f2ce077656e940baf00946a", element="f.E5D17DD505834005EF496BBDFC6DDF12.d.06C69BAA94BA824E74CEB1B6B2D6B90A.e.31")>]
['T Ramos', 'FB', nan, 'home', 1.0, 0.0, 3.0, 3.0, 0.0, 20.0]
['p_name', 'position', 'p_id', 'team', 'tries', 'try_assists', 'conversion_goals', 'penalty_goals', 'drop_goals_converted', 'points']
['B Kinghorn', 'FB', nan, 'away', 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
['p_name', 

,p_name,position,p_id,team,tries,try_assists,conversion_goals,penalty_goals,drop_goals_converted,points,...,offloads,to_conceded,tackles,missed_tackles,lineouts_won,penalties_conceded,yellows,reds,game_id,league_df
0,T Ramos,FB,NaN,home,1.0,0.0,3.0,3.0,0.0,20.0,...,0.0,0.0,4.0,3.0,0.0,0.0,0.0,0.0,600264,180659
1,D Penaud,W,NaN,home,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,5.0,3.0,0.0,0.0,0.0,0.0,600264,180659
2,G Fickou,C,NaN,home,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,1.0,10.0,5.0,0.0,1.0,0.0,0.0,600264,180659
3,Y Moefana,C,NaN,home,2.0,0.0,0.0,0.0,0.0,10.0,...,0.0,0.0,13.0,5.0,0.0,0.0,0.0,0.0,600264,180659
4,L Bielle-Biarrey,W,NaN,home,1.0,0.0,0.0,0.0,0.0,5.0,...,1.0,0.0,1.0,1.0,0.0,2.0,0.0,0.0,600264,180659
5,R Ntamack,FH,NaN,home,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,11.0,0.0,0.0,1.0,0.0,0.0,600264,180659
6,M Lucu,SH,NaN,home,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,12.0,0.0,0.0,0.0,0.0,0.0,600264,180659
7,J Gros,P,NaN,home,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,8.0,1.0,0.0,0.0,1.0,0.0,600264,180659
8,P Mauvaka,H,NaN,home,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,5.0,2.0,6.0,1.0,1.0,0.0,600264,180659
9,U Atonio,P,NaN,home,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,600264,180659


In [8]:
match_scrape.close_out_driver()